In [ ]:
# Goals
# 1. Impact Weighting posts by engagement
# 2. Cross-stock sentiment infuence
# 3. Differeniation between generic and stock-specific sentiment signals
# 4. Compare FinBERT vs CryptoBERT vs DeepSeek
# 5. Market Regimes influence - volatility, bull vs bear, crisis vs normal
# 6. Sentiment Esemble model vs DDQN vs Baselines
# 7. Ablation Study - removing one component at a time to see its impact on performance (with/without sentiment, with/without technical indicators)
# 8. Posts count per day influence (100vs10)
# 9. Compare WinRate, SharpeRatio, Cumulative Returns


In [ ]:
import mlflow

mlflow.end_run()

In [ ]:
import import_ipynb
import pandas as pd
import numpy as np
from collections import deque
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, InputLayer, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import random
import time 
import tensorflow as tf
from Data_Preprocessing import Data_Preprocessing
from Calculate_Returns import Calculate_Returns
from Calculate_Returns_Simple import Calculate_Returns_Simple
from Technical_Indicators import Technical_Indicators
from Triple_Barrier_Labelel import Triple_Barrier_Labelel
from Model_Train import Model_Train
import torch
import os
import gc
from Association_Rule_Mining import create_tweet_dataset, create_continous_dataset, create_triple_barrier_labeling, create_categorize_dataset
from math import sqrt
import matplotlib.pyplot as plt
import mlflow
from tqdm.auto import tqdm

DEBUG = False

dp = Data_Preprocessing()
mt = Model_Train('', '')
encoder = OneHotEncoder(sparse=False)
scaler = MinMaxScaler()
tbl = Triple_Barrier_Labelel()

MEMORY_LENGTH = 100
BATCH_SIZE = 64
MODEL_DESIGN = "64/64"
LEARNING_RATE = 0.0005
FEE = 0.0005
INITIAL_CASH = 100000
TARGET_UPDATE = 20
EPISODES = 50
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.99
GAMMA = 0.95
EPSILON = 1.0
TWEETS_RANDOM_SAMPLE = True
TWEETS_ENGAGEMENT_POSTS = False
TWEETS_DAILY_SAMPLE_SIZE = 50
TWEETS_RANDOM_SEED = 42
TEST_REPEATS = 5
SCALE = 50

STARTING_DATE = "2018-01-01"
ENDING_DATE ="2019-06-01"

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
def evaluate_model(original_data, scaled_data, mod):
    print("Evaluating...")
    returns = Calculate_Returns_Simple(pd.Series([x[0] for x in original_data]), FEE, INITIAL_CASH)
    profits = []
    action = None
    
    for step, _ in enumerate(original_data[:-1]):
        offset = step
        state = returns.append_context_and_unrealized_pnl_to_state(scaled_data[offset], offset)
        q_values = mod.predict(np.array(state).reshape(1,-1), verbose=0)
        action = np.argmax(q_values[0])
        offset += 1
        reward = returns.perform_action(action, offset)
        profits.append(reward)
    returns.close_last_position(original_data[-1])
    return returns

In [ ]:
class Agent():
    def __init__(self, action_size, state_size, gamma, epsilon, epsilon_min, epsilon_decay):
        self.action_size = action_size
        self.state_size = state_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=MEMORY_LENGTH)
        self.model = self.create_model()
        self.target_model = clone_model(self.model)
        self.optimizer = Adam(learning_rate=LEARNING_RATE) 
        self.step_counter = 0
        
    def generate_model(self, state_size, action_size):
        layers = MODEL_DESIGN.split("/")
        model = Sequential()
        state_size = state_size
        model.add(InputLayer(input_shape=(state_size,), name="InputLayer"))
        for index, units in enumerate(layers):
            model.add(Dense(units=units, activation="relu", name=f"HiddenLayer{index}"))
            model.add(Dropout(0.1))
        model.add(Dense(units=action_size, activation='linear', name="OutputLayer"))
        return model

    def create_model(self):
        return self.generate_model(self.state_size, self.action_size)

    def act(self, state):
        if random.uniform(0,1) < self.epsilon:
            rand_action = random.randrange(self.action_size)
            return rand_action
        state = np.array(state).reshape(1, -1)
        q_values = self.model(np.array(state, dtype=np.float32), training=False).numpy()
        return np.argmax(q_values[0])
    
    def remember(self, state, action, reward, new_state, done):
        self.memory.append((state, action, reward, new_state, done))
        
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())
    
    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return
        minibatch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, new_states, done = zip(*minibatch)
        states = np.array(states, dtype=np.float32)
        new_states = np.array(new_states, dtype=np.float32)
        actions = np.array(actions, dtype=np.int32)
        rewards = np.array(rewards, dtype=np.float32)
        done = np.array(done, dtype=np.float32)

        best_action_indices = np.argmax(self.model(new_states, training=False).numpy(), axis=1)
        target = rewards + (1 - done) * self.gamma * self.target_model(new_states, training=False).numpy()[np.arange(BATCH_SIZE), best_action_indices]
        with tf.GradientTape() as tape:
            current_Q_values = self.model([states], training=True)
            action_mask = tf.one_hot(actions, current_Q_values.shape[1])
            predicted_Q_values = tf.reduce_sum(current_Q_values * action_mask, axis=1)
            loss = tf.keras.losses.MeanSquaredError()(target, predicted_Q_values)
        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.model.trainable_variables))
        self.step_counter += 1
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        if self.step_counter % TARGET_UPDATE == 0:
            self.step_counter = 0
            self.update_target_model()
        del states, actions, rewards, new_states, done, minibatch
        gc.collect()

In [ ]:
class Environment():
    def __init__(self, data, data_scaled):
        self.action_size = 3
        self.state_size = np.array(data_scaled).shape[1] + np.zeros(3).shape[0] + 1 #Contatenate the state with the position one-hot encoding and unrealized PnL
        self.data = data
        self.data_scaled = data_scaled
        self.offset = 0
        self.steps = len(data)
        self.returns = Calculate_Returns_Simple(pd.Series([x[0] for x in self.data]), FEE, INITIAL_CASH)
        
    
    def step(self, action):
        self.offset = self.offset + 1
        new_state = self.data_scaled[self.offset]
        new_state = self.returns.append_context_and_unrealized_pnl_to_state(new_state, self.offset)
        reward = self.returns.perform_action(action, self.offset)
        done = self.offset == len(self.data) - 1
        # print(f"Price: {self.data[self.offset][0]}, Step: {self.offset}, Action: {action}, Reward: {reward}, Done: {done}")
        return new_state, reward, done
    
    def reset(self):
        self.offset = 0
        self.context = 0
        self.returns = Calculate_Returns_Simple(pd.Series([x[0] for x in self.data]), FEE, INITIAL_CASH)
        state_only = self.data_scaled[self.offset]
        state_only = self.returns.append_context_and_unrealized_pnl_to_state(state_only, 0)
        return state_only

In [ ]:
class DQNAlgorithm():
    def __init__(self, data, data_scaled, episodes, gamma, epsilon, epsilon_min, epsilon_decay):
        self.episodes = episodes
        self.data = data
        self.data_scaled = data_scaled
        self.env = Environment(data, data_scaled)
        self.steps = self.env.steps
        self.agent = Agent(self.env.action_size, self.env.state_size, gamma, epsilon, epsilon_min, epsilon_decay)
        self.best_model_win_rate = 0
        encoder.fit([[0], [1], [2]])
        self.best_models = []
        
    def debug(self):
        print("Debugging...")
        state = self.env.reset()
        profits = []
        for step in range(self.steps):
            print(f"Step: {step+1}/{self.steps}")
            print(f"Price: {self.data[step][0]}")
            print(f"Context: {self.env.returns.context}")
            # print(f"State: {state}")
            action = self.agent.act(state)
            print(f"Action: {action}")
            new_state, reward, done = self.env.step(action)
            print(f"New Context: {self.env.returns.context}")
            print(f"Next Price: {self.data[step + 1][0]}")
            print(f"Reward: {reward}")
            # print(f"New State: {new_state}")
            self.agent.remember(state, action, reward, new_state, done)
            profits.append(reward)
            print(f"--------------------------------------------------------------------------")
            if done == True:
                break
            self.agent.replay()
            state = new_state

    def run(self):
        for episode in tqdm(range(self.episodes), desc="DDQN Model Training"):
            state = self.env.reset()
            actions = 0
            for step in range(self.steps):
                actions+=1
                action = self.agent.act(state)
                new_state, reward, done = self.env.step(action)
                self.agent.remember(state, action, reward, new_state, done)
                if done == True:
                    self.env.returns.close_last_position(self.data[-1][0])
                    new_model = clone_model(self.agent.model)
                    new_model.set_weights(self.agent.model.get_weights())
                    self.best_models.append(new_model)
                    break
                self.agent.replay()
                state = new_state
        print("Cleaning memory...")
        gc.collect()
        tf.keras.backend.clear_session()
        return self.best_models

In [ ]:
dp = Data_Preprocessing()
ti = Technical_Indicators()
mlflow.end_run()

cases = pd.read_csv("../TestCases/SE_DDQN_Test_Cases.csv")
for test_id in range(1, 7):
    test_type = "TechnicalOnly" #cases["TestType"][test_id]
    market = cases["Market"][test_id]
    param = cases["Param"][test_id]

    market_data = pd.read_csv(f"../Datasets/LabeledDatasets/TBL_Labels_{market}.csv")
    market_data.set_index("timestamp", inplace=True)
    market_data = market_data.dropna()
    if test_type == "CrossStock":
        sentiment_data = pd.read_csv(f"../Predictions/{param}_predictions.csv")
    else:
        sentiment_data = pd.read_csv(f"../Predictions/{market}_predictions.csv")
    sentiment_data.set_index("day", inplace=True)
    sentiment_data = pd.get_dummies(sentiment_data, columns=["predicted_class"])
    merged = dp.merge_data(sentiment_data, market_data, False)
    merged = dp.merge_data(merged, ti.build_market_features(merged['close'], merged['volume']).dropna(), False)

    if test_type == "NoTechnicalIndicators":
        params = "predicted_class_0,predicted_class_1,predicted_class_2,confidence"
    elif test_type == "TechnicalOnly":
        params = "ROC_direction,RSI_Momentum_exhaustion,SMA_Trend_structure,MACD_Trend_acceleration,Level_ZScore_Market_tension,Ret_ZScore_Price_shock,Volatility_Market_risk,Volatility_Structure,Liquidity_Pressure"
    else:
        params = "ROC_direction,RSI_Momentum_exhaustion,SMA_Trend_structure,MACD_Trend_acceleration,Level_ZScore_Market_tension,Ret_ZScore_Price_shock,Volatility_Market_risk,Volatility_Structure,Liquidity_Pressure,predicted_class_0,predicted_class_1,predicted_class_2,confidence"

    sel_param = ["close"] + [el.strip() for el in params.split(",")]
    

    existing_cols = [col for col in sel_param if col in merged.columns]
    merged_data = merged[existing_cols]

    merged_data = merged_data.values.tolist()
    n = len(merged_data)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    train_data = merged_data[:train_end]
    val_data = merged_data[train_end:val_end]
    test_data = merged_data[val_end:]
    train_x = np.array(train_data)[:, 1:]
    train_y = np.array(train_data)[:, :1]
    val_x = np.array(val_data)[:, 1:]
    val_y = np.array(val_data)[:, :1]
    test_x = np.array(test_data)[:, 1:]
    test_y = np.array(test_data)[:, :1]

    scaler.fit(train_x)
    train_scaled = scaler.transform(train_x).tolist()
    val_scaled = scaler.transform(val_x).tolist()
    test_scaled = scaler.transform(test_x).tolist()
    
    test_results_df = []
    
    dqn = DQNAlgorithm(data=train_y, data_scaled=train_scaled, episodes=EPISODES, gamma=GAMMA, epsilon=EPSILON, epsilon_min=EPSILON_MIN, epsilon_decay = EPSILON_DECAY)

    if DEBUG == True:
        print(f"Debug Mode: Running {test_type} ({rep}) on {market} with parameters: {params}")
        dqn.debug()
        break
    else:
        mlflow.set_experiment(market + "_" + test_type)
        with mlflow.start_run():
            mlflow.log_param("model", "DNN")
            mlflow.log_param("market", market)
            mlflow.log_param("episodes", EPISODES)
            mlflow.log_param("test_repeats", TEST_REPEATS)
            mlflow.log_param("layers", MODEL_DESIGN)
            mlflow.log_param("optimizer", "Adam")
            mlflow.log_param("classes", "Hold, Short, Long")
            mlflow.log_param("parameters", params)
            mlflow.log_param("sentiment", "No" if test_type == "TechnicalOnly" else f"{market} 100 Posts Per Day")
            mlflow.log_param("memory_length", MEMORY_LENGTH)
            mlflow.log_param("batch_size", BATCH_SIZE)
            mlflow.log_param("target_update", TARGET_UPDATE)
            mlflow.log_param("gamma", GAMMA)
            mlflow.log_param("epsilon_min", EPSILON_MIN)
            mlflow.log_param("epsilon_decay", EPSILON_DECAY)
            mlflow.log_param("learning_rate", LEARNING_RATE)
            
            for rep in range(0, TEST_REPEATS):
                try:
                    best_models = dqn.run()
                    period = 365 if market == "BTC" else 252
                    
                    val_results_df = pd.DataFrame([], columns=["Model_Index", "Sharpe-Ratio", "Calmar-Ratio", "Transactions"])
                    for index, model in enumerate(best_models):
                        returns = evaluate_model(val_y, val_scaled, model)

                        sharpe = returns.sharpe(period=period)
                        calmar_ratio = returns.calmar_ratio()

                        val_results_df.loc[len(val_results_df)] = [
                            index,
                            sharpe,
                            calmar_ratio,
                            len(returns.records)
                        ]            

                    val_results_df["Score"] = val_results_df["Sharpe-Ratio"] + 0.5 * val_results_df["Calmar-Ratio"]

                    models_to_test = val_results_df[val_results_df["Transactions"] > 0]\
                        .sort_values(by="Score", ascending=False)\
                        .head(5)

                    if len(models_to_test) == 0:
                        raise Exception("No sufficient models found!")

                    for _, model_to_test in models_to_test.iterrows():

                        model_index = int(model_to_test["Model_Index"])
                        model = best_models[model_index]

                        returns = evaluate_model(test_y, test_scaled, model)
                        test_results_df.append({
                            "Test_Type": test_type,
                            "Market": market,
                            "Model_Index": model_index,
                            "Sharpe-Ratio": returns.sharpe(period = period),
                            "Win-Rate": returns.win_rate(),
                            "Sortino": returns.sortino(period = period),
                            "Max-Drawdown": returns.max_drawdown(),
                            "Annualized-Return": returns.annualized_return(period = period),
                            "Calmar-Ratio": returns.calmar_ratio(),
                            "Cumulative Return": returns.cumulative_return(),
                            "Transactions": len(returns.records),
                            "Holds": len([record for record in returns.records if record['action'] == 0]),
                            "Shorts": len([record for record in returns.records if record['action'] == 1]),
                            "Longs": len([record for record in returns.records if record['action'] == 2])
                        })
                        model_name = f"DNN_{test_type}_{market}_100PostsPerDay_{model_index}"
                        mlflow.tensorflow.log_model(model, name=model_name, input_example=np.zeros((1, train_x.shape[1] + 4)))
                    
                except Exception as e:
                    print(f"Error in repetition {rep}: {e}")

            if len(test_results_df) > 0:
                mean_test_sharpe = np.mean([result["Sharpe-Ratio"] for result in test_results_df])
                std_test_sharpe = np.std([result["Sharpe-Ratio"] for result in test_results_df])
                mean_test_calmar = np.mean([result["Calmar-Ratio"] for result in test_results_df])
                std_test_calmar = np.std([result["Calmar-Ratio"] for result in test_results_df])
                mean_test_win_rate = np.mean([result["Win-Rate"] for result in test_results_df])
                std_test_win_rate = np.std([result["Win-Rate"] for result in test_results_df])
                mean_test_sortino = np.mean([result["Sortino"] for result in test_results_df])
                std_test_sortino = np.std([result["Sortino"] for result in test_results_df])
                mean_test_max_drawdown = np.mean([result["Max-Drawdown"] for result in test_results_df])
                std_test_max_drawdown = np.std([result["Max-Drawdown"] for result in test_results_df])
                mean_test_annualized_return = np.mean([result["Annualized-Return"] for result in test_results_df])
                std_test_annualized_return = np.std([result["Annualized-Return"] for result in test_results_df])
                mean_test_cumulative_return = np.mean([result["Cumulative Return"] for result in test_results_df])
                std_test_cumulative_return = np.std([result["Cumulative Return"] for result in test_results_df])
                mean_test_transactions = np.mean([result["Transactions"] for result in test_results_df])
                std_test_transactions = np.std([result["Transactions"] for result in test_results_df])
                mean_test_holds = np.mean([result["Holds"] for result in test_results_df])
                std_test_holds = np.std([result["Holds"] for result in test_results_df])
                mean_test_shorts = np.mean([result["Shorts"] for result in test_results_df])
                std_test_shorts = np.std([result["Shorts"] for result in test_results_df])
                mean_test_longs = np.mean([result["Longs"] for result in test_results_df])
                std_test_longs = np.std([result["Longs"] for result in test_results_df])
                print(f"{market}, {test_type}: Logging the metrics across all tested models...")
                mlflow.log_metric("Win-Rate Mean", mean_test_win_rate)
                mlflow.log_metric("Win-Rate Std", std_test_win_rate)
                mlflow.log_metric("Sharpe Mean", mean_test_sharpe)
                mlflow.log_metric("Sharpe Std", std_test_sharpe)
                mlflow.log_metric("Calmar Mean", mean_test_calmar)
                mlflow.log_metric("Calmar Std", std_test_calmar)
                mlflow.log_metric("Sortino Mean", mean_test_sortino)
                mlflow.log_metric("Sortino Std", std_test_sortino)
                mlflow.log_metric("Max-Drawdown Mean", mean_test_max_drawdown)
                mlflow.log_metric("Max-Drawdown Std", std_test_max_drawdown)
                mlflow.log_metric("Annualized-Return Mean", mean_test_annualized_return)
                mlflow.log_metric("Annualized-Return Std", std_test_annualized_return)
                mlflow.log_metric("Cumulative-Return Mean", mean_test_cumulative_return)
                mlflow.log_metric("Cumulative-Return Std", std_test_cumulative_return)
                mlflow.log_metric("Transactions Mean", mean_test_transactions)
                mlflow.log_metric("Transactions Std", std_test_transactions)
                mlflow.log_metric("Holds Mean", mean_test_holds)
                mlflow.log_metric("Holds Std", std_test_holds)
                mlflow.log_metric("Shorts Mean", mean_test_shorts)
                mlflow.log_metric("Shorts Std", std_test_shorts)
                mlflow.log_metric("Longs Mean", mean_test_longs)
                mlflow.log_metric("Longs Std", std_test_longs)
                pd.DataFrame(test_results_df).to_csv("test_results.csv", index=False)
                mlflow.log_artifact("test_results.csv")
                mlflow.end_run()

            else:
                print(f"{market}, {test_type}: Error No valid test results found, skipping metric logging!")
                mlflow.end_run()
                break

In [ ]:
# test_results = {
#     "Sharpe-Ratio": -np.inf,
# }
# best_model = None

In [ ]:
# #Validation
# for index, model in enumerate(best_models):
#     returns = evaluate_model(train_y, train_scaled, model)
#     sharpe = returns.sharpe(period = period)
#     if  sharpe > test_results["Sharpe-Ratio"]:
#         test_results["Sharpe-Ratio"] = sharpe
#         best_model = model
    
# #Testing
# returns = evaluate_model(test_y, test_scaled, best_model)

#Save testing results to mlflow
# mlflow.log_metric("Win-Rate", returns.win_rate())
# mlflow.log_metric("Sharpe-Ratio", returns.sharpe(period = period))
# mlflow.log_metric("Sortino", returns.sortino(period = period))
# mlflow.log_metric("Max-Drawdown", returns.max_drawdown())
# mlflow.log_metric("Annualized-Return", returns.annualized_return(period = period))
# mlflow.log_metric("Calmar-Ratio", returns.calmar_ratio())
# mlflow.log_metric("Cumulative Return", returns.cumulative_return())
        
# equity_curve_df = pd.DataFrame({
#     "step": np.arange(len(returns.equity_curve())),
#     "win_rate": returns.equity_curve()
# })
# equity_curve_df.to_csv("ec_iterations.csv", index=False)
# mlflow.log_artifact("ec_iterations.csv")

# train_results_df = []
# for index, model in enumerate(best_models):
#     returns = evaluate_model(train_y, train_scaled, model)
#     sharpe = returns.sharpe(period = period)
#     win_rate = returns.win_rate()
#     sortino = returns.sortino(period = period)
#     max_drawdown = returns.max_drawdown()
#     annualized_return = returns.annualized_return(period = period)
#     calmar_ratio = returns.calmar_ratio()
#     cumulative_return = returns.cumulative_return()
#     train_results_df.append({
#         "Model_Index": index,
#         "Sharpe-Ratio": sharpe,
#         "Win-Rate": win_rate,
#         "Sortino": sortino,
#         "Max-Drawdown": max_drawdown,
#         "Annualized-Return": annualized_return,
#         "Calmar-Ratio": calmar_ratio,
#         "Cumulative Return": cumulative_return,
#         "Transactions": len(returns.records),
#         "Holds": len([record for record in returns.records if record['action'] == 0]),
#         "Shorts": len([record for record in returns.records if record['action'] == 1]),
#         "Longs": len([record for record in returns.records if record['action'] == 2])
#     })
    
# train_tx_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "transactions": [row['Transactions'] for row in train_results_df]
# })
# train_tx_curve_df.to_csv("train_tx_iterations.csv", index=False)
# mlflow.log_artifact("train_tx_iterations.csv")

# train_holds_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "holds": [row['Holds'] for row in train_results_df]
# })
# train_holds_curve_df.to_csv("train_holds_iterations.csv", index=False)
# mlflow.log_artifact("train_holds_iterations.csv")

# train_shorts_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "shorts": [row['Shorts'] for row in train_results_df]
# })
# train_shorts_curve_df.to_csv("train_shorts_iterations.csv", index=False)
# mlflow.log_artifact("train_shorts_iterations.csv")

# train_longs_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "longs": [row['Longs'] for row in train_results_df]
# })
# train_longs_curve_df.to_csv("train_longs_iterations.csv", index=False)
# mlflow.log_artifact("train_longs_iterations.csv")

# train_wr_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "win_rate": [row['Win-Rate'] for row in train_results_df]
# })
# train_wr_curve_df.to_csv("train_wr_iterations.csv", index=False)
# mlflow.log_artifact("train_wr_iterations.csv")

# train_sharpe_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "sharpe_ratio": [row['Sharpe-Ratio'] for row in train_results_df]
# })
# train_sharpe_curve_df.to_csv("train_sharpe_iterations.csv", index=False)
# mlflow.log_artifact("train_sharpe_iterations.csv")

# train_sortino_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "sortino_ratio": [row['Sortino'] for row in train_results_df]
# })
# train_sortino_curve_df.to_csv("train_sortino_iterations.csv", index=False)
# mlflow.log_artifact("train_sortino_iterations.csv")

# train_max_drawdown_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "max_drawdown": [row['Max-Drawdown'] for row in train_results_df]
# })
# train_max_drawdown_curve_df.to_csv("train_max_drawdown_iterations.csv", index=False)
# mlflow.log_artifact("train_max_drawdown_iterations.csv")

# train_annualized_return_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "annualized_return": [row['Annualized-Return'] for row in train_results_df]
# })
# train_annualized_return_curve_df.to_csv("train_annualized_return_iterations.csv", index=False)
# mlflow.log_artifact("train_annualized_return_iterations.csv")

# train_calmar_ratio_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "calmar_ratio": [row['Calmar-Ratio'] for row in train_results_df]
# })
# train_calmar_ratio_curve_df.to_csv("train_calmar_ratio_iterations.csv", index=False)
# mlflow.log_artifact("train_calmar_ratio_iterations.csv")

# train_cumulative_return_curve_df = pd.DataFrame({
#     "step": np.arange(len(train_results_df)),
#     "cumulative_return": [row['Cumulative Return'] for row in train_results_df]
# })
# train_cumulative_return_curve_df.to_csv("train_cumulative_return_iterations.csv", index=False)
# mlflow.log_artifact("train_cumulative_return_iterations.csv")

In [ ]:

# if best_model is None:
#     print("No valid model found, skipping log_model")
# else:
#     model_name = f"DNN_{test_type}_{market}_100PostsPerDay"
#     mlflow.tensorflow.log_model(best_model, name=model_name, input_example=np.zeros((1, train_x.shape[1])))

In [ ]:
# cases = pd.read_csv("../TestCases/DDQN_Test_Cases.csv")
# for test_id in range(2, 6):
#     params = cases["Parameters"][test_id]
#     market = cases["Market"][test_id]
#     name = cases["TestCaseName"][test_id]
    
#     continous_df = create_continous_dataset(market, starting_date=STARTING_DATE, ending_date=ENDING_DATE)
#     continous_df_with_tbl = create_triple_barrier_labeling(continous_df)
#     tweets_df = create_tweet_dataset("../Datasets/investing_classified_sentiments" if market != "BTC-USD" else "../Datasets/btc_classified_sentiments") #"Datasets/investing_classified_sentiments"
#     merged_df = pd.concat([continous_df_with_tbl, tweets_df], axis=1).dropna()
#     # extended = merged_df #extend_time_columns(merged_df, skip_cols=["signals"], t=7)
#     categorized_for_trade_profitable = create_categorize_dataset(merged_df, suffix_vals=['bearish', 'Bearish', 'bullish', 'Bullish', '1.0', '1', '0.0', '0', '-1', '-1.0'], skip_cols=["next_day_label", "signals", "previous_label"])
#     merged_df = pd.concat([merged_df, categorized_for_trade_profitable], axis=1).dropna()

#     sel_param = ["close"] + [el.strip() for el in params.split(",")]
    
#     existing_cols = [col for col in sel_param if col in merged_df.columns]
#     merged_data = merged_df[existing_cols]
    
#     merged_data = merged_data.values.tolist()
#     train_end = int(len(merged_data) * 0.8)
#     train_data = merged_data[:train_end]
#     test_data  = merged_data[train_end:]
#     train_x = np.array(train_data)[:,1:]
#     train_y = np.array(train_data)[:,:1]
#     test_x = np.array(test_data)[:,1:]
#     test_y = np.array(test_data)[:,:1]
#     scaler.fit(train_x)
#     train_scaled = scaler.transform(train_x).tolist()
#     test_scaled = scaler.transform(test_x).tolist()
    
#     test_results_df = []
    
#     for rep in range(0, TEST_REPEATS):
#         mlflow.set_experiment(f"{name}_return_classification_v2")
#         mlflow.start_run()
#         mlflow.log_param("model", "DNN")
#         mlflow.log_param("test_number", rep)
#         mlflow.log_param("market", market)
#         mlflow.log_param("episodes", EPISODES)
#         mlflow.log_param("layers", MODEL_DESIGN)
#         mlflow.log_param("optimizer", "Adam")
#         mlflow.log_param("classes", "Hold, Short, Long")
#         mlflow.log_param("parameters", params)
#         mlflow.log_param("memory_length", MEMORY_LENGTH)
#         mlflow.log_param("batch_size", BATCH_SIZE)
#         mlflow.log_param("target_update", TARGET_UPDATE)
#         mlflow.log_param("gamma", GAMMA)
#         mlflow.log_param("epsilon_min", EPSILON_MIN)
#         mlflow.log_param("epsilon_decay", EPSILON_DECAY)
#         mlflow.log_param("learning_rate", LEARNING_RATE)

#         dqn = DQNAlgorithm(data=train_y, data_scaled=train_scaled, episodes=EPISODES, gamma=GAMMA, epsilon=EPSILON, epsilon_min=EPSILON_MIN, epsilon_decay = EPSILON_DECAY)
#         if DEBUG == True:
#             dqn.debug(train_data)
#             break
#         else:
#             best_models = dqn.run()

#         test_results = {
#             "Win-Rate": -np.inf,
#             "Sharpe-Ratio": -np.inf,
#             "Cum-Returns": -np.inf,
#         }
#         best_model = None
#         win_rates_over_iterations = []
#         sharpe_ratios_over_iterations = []
#         cum_returns_over_iterations = []
        
#         for index, model in enumerate(best_models):
#             score = evaluate_model(test_y, test_scaled, model, False, market)
#             win_rates_over_iterations.append(score["Win-Rate"])
#             sharpe_ratios_over_iterations.append(score["Sharpe-Ratio"])
#             cum_returns_over_iterations.append(score["Cum-Returns"][-1])
#             if score["Win-Rate"] > test_results["Win-Rate"]:
#                 test_results = score
#                 best_model = model
        
#         win_rates_iterations_df = pd.DataFrame({
#             "step": np.arange(len(win_rates_over_iterations)),
#             "win_rate": win_rates_over_iterations
#         })
#         win_rates_iterations_df.to_csv("wr_iterations.csv", index=False)
#         mlflow.log_artifact("wr_iterations.csv")
        
#         sharpe_iterations_df = pd.DataFrame({
#             "step": np.arange(len(sharpe_ratios_over_iterations)),
#             "sharpe_ratio": sharpe_ratios_over_iterations
#         })
#         sharpe_iterations_df.to_csv("sharpe_iterations.csv", index=False)
#         mlflow.log_artifact("sharpe_iterations.csv")
        
#         cr_iterations_df = pd.DataFrame({
#             "step": np.arange(len(cum_returns_over_iterations)),
#             "cumulative_return": cum_returns_over_iterations
#         })
#         cr_iterations_df.to_csv("cum_returns_iterations.csv", index=False)
#         mlflow.log_artifact("cum_returns_iterations.csv")

#         mlflow.log_metric("Win-Rate", test_results["Win-Rate"])
#         mlflow.log_metric("Sharpe-Ratio", test_results["Sharpe-Ratio"])
#         cum_df = pd.DataFrame({ 
#             "step": np.arange(len(test_results["Cum-Returns"])),
#             "cumulative_return": test_results["Cum-Returns"]
#         })
#         cum_df.to_csv("cum_returns.csv", index=False)
#         mlflow.log_artifact("cum_returns.csv")
#         if best_model is None:
#             print("⚠ No valid model found, skipping log_model")
#         else:
#             model_name = f"{name}_{rep}"
#             mlflow.tensorflow.log_model(best_model, name=model_name, input_example=np.zeros((1, train_x.shape[1])))
#         mlflow.end_run()

    # test_results_df = pd.DataFrame(test_results_df)
    # columns = ["TestCaseName", "Test Repeats", "Train Episodes", "Market", "Parameters", "Positive Rewards", "Wins", "Sum PnL", "Sharpe Ratio"]
    # test_conc = [name, TEST_REPEATS, EPISODES, market, params, f"Mean: {test_results_df['Positive Rewards'].mean()}, Std: {test_results_df['Positive Rewards'].std()}", f"Mean: {test_results_df['Wins'].mean()}, Std: {test_results_df['Wins'].std()}", f"Mean: {pd.to_numeric(test_results_df['Sum PnL'], errors='coerce').mean()}, Std: {pd.to_numeric(test_results_df['Sum PnL'], errors='coerce').std()}", f"Mean: {test_results_df['Sharpe Ratio'].mean()}, Std: {test_results_df['Sharpe Ratio'].std()}"]
    # pd.DataFrame([test_conc], columns=columns).to_csv(f"TestResults/DDQN_Test_Results.csv", mode="a", index=False, header=not os.path.exists("TestResults/DDQN_Test_Results.csv"))
